# Aula 4 — Controle de qualidade e trimming

Esta prática começa exatamente onde a Aula 3 terminou: nos FASTQ paired-end salvos em
`03_sra_fastq/raw-fastq/`.

## 0. Preparar o runtime

O Google Drive guarda os **dados** entre as aulas, mas o runtime do Colab é temporário.
Programas instalados no runtime podem desaparecer quando a sessão termina.

Por isso, quando uma aula precisar de ferramentas externas, começaremos verificando se
o Conda já existe. Se não existir, ele será instalado antes de qualquer outra configuração.

> Esta deve ser a primeira célula executável do notebook, porque a instalação do Conda
> pode reiniciar o runtime.

In [ ]:
import shutil

if shutil.which("conda"):
    print("Conda já está disponível neste runtime.")
else:
    !pip install -q condacolab
    import condacolab
    condacolab.install()

### Verificar o Conda e configurar Bioconda

Usaremos a configuração recomendada pelo Bioconda: `conda-forge` com maior prioridade,
seguido de `bioconda`, e prioridade estrita.

Como `conda config --add` adiciona canais do menor para o maior nível de prioridade,
executamos primeiro `bioconda` e depois `conda-forge`.

In [ ]:
!conda --version
!conda config --remove-key channels 2>/dev/null || true
!conda config --add channels bioconda
!conda config --add channels conda-forge
!conda config --set channel_priority strict
!conda config --show channels

## 1. Retomar o projeto no Google Drive

Todas as práticas usam a mesma raiz:

`/content/drive/MyDrive/Bioinformatica_Biologia_Molecular`

Os resultados de uma aula são lidos pela aula seguinte. Assim, os **dados persistem**
mesmo quando o runtime do Colab é encerrado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

ROOT = Path("/content/drive/MyDrive/Bioinformatica_Biologia_Molecular")
RUN = "SRR15736591"
SAMPLE = "hypochilus_petrunkevitchi_SRR15736591"

PASTAS = {
    "01_bancos": ROOT / "01_bancos",
    "02_blast": ROOT / "02_blast",
    "03_raw": ROOT / "03_sra_fastq" / "raw-fastq",
    "04_qc": ROOT / "04_qc_trimming",
    "04_trimmed": ROOT / "04_qc_trimming" / "trimmed",
    "05_assemblies": ROOT / "05_spades" / "spades-assemblies",
    "05_contigs": ROOT / "05_spades" / "spades-assemblies" / "contigs",
    "06_match": ROOT / "06_uce_match",
    "06_probes": ROOT / "06_uce_match" / "probes",
    "06_results": ROOT / "06_uce_match" / "uce-search-results",
    "07_taxon_sets": ROOT / "07_uce_extract" / "taxon-sets" / "all",
    "08_integracao": ROOT / "08_integracao",
    "ambientes": ROOT / "ambientes",
}

for pasta in PASTAS.values():
    pasta.mkdir(parents=True, exist_ok=True)

os.chdir(ROOT)

print("Diretório atual:", Path.cwd())
print("\nEstrutura principal do projeto:")
for chave, pasta in PASTAS.items():
    print(f"{chave:15s} -> {pasta.relative_to(ROOT)}")

## 2. Verificar os arquivos produzidos na Aula 3

In [ ]:
RAW = PASTAS["03_raw"]
OUT = PASTAS["04_qc"]
TRIMMED = PASTAS["04_trimmed"]

R1 = RAW / f"{RUN}_1.fastq.gz"
R2 = RAW / f"{RUN}_2.fastq.gz"

for arquivo in [R1, R2]:
    print(arquivo, "->", "OK" if arquivo.exists() else "AUSENTE")

if not R1.exists() or not R2.exists():
    raise FileNotFoundError(
        "FASTQ da Aula 3 não encontrado. Execute primeiro 03_FASTA_FASTQ_SRA.ipynb."
    )

print("\nEntrada:", RAW)
print("Saída  :", OUT)

## 3. Preparar o ambiente e instalar as ferramentas necessárias

In [ ]:
!conda env list | grep -qE '^bioinfo[[:space:]]' || conda create -y -n bioinfo python=3.11
!conda install -y -n bioinfo fastqc multiqc trimmomatic
!conda run -n bioinfo fastqc --version
!conda run -n bioinfo multiqc --version
!conda run -n bioinfo trimmomatic -version

## 4. FastQC nos reads brutos

In [ ]:
raw_qc = OUT / "fastqc_raw"
raw_qc.mkdir(parents=True, exist_ok=True)

!conda run -n bioinfo fastqc -t 2 -o "$raw_qc" "$R1" "$R2"
!ls -lh "$raw_qc"

## 5. MultiQC dos dados brutos

In [ ]:
multi_raw = OUT / "multiqc_raw"
multi_raw.mkdir(parents=True, exist_ok=True)

!conda run -n bioinfo multiqc "$raw_qc" -o "$multi_raw" -f

## 6. Localizar o arquivo de adaptadores do Trimmomatic

In [ ]:
import subprocess
from pathlib import Path

prefix = subprocess.check_output(
    ["conda", "run", "-n", "bioinfo", "bash", "-lc", "printf %s \"$CONDA_PREFIX\""],
    text=True
).strip()

candidatos = list(Path(prefix).rglob("TruSeq3-PE.fa"))
if not candidatos:
    raise FileNotFoundError("TruSeq3-PE.fa não foi encontrado dentro do ambiente bioinfo.")

adapter = candidatos[0]
print("Adapters:", adapter)

## 7. Executar Trimmomatic paired-end

In [ ]:
R1P = TRIMMED / f"{SAMPLE}_R1_paired.fastq.gz"
R1U = TRIMMED / f"{SAMPLE}_R1_unpaired.fastq.gz"
R2P = TRIMMED / f"{SAMPLE}_R2_paired.fastq.gz"
R2U = TRIMMED / f"{SAMPLE}_R2_unpaired.fastq.gz"

!conda run -n bioinfo trimmomatic PE -threads 2 -phred33   "$R1" "$R2"   "$R1P" "$R1U"   "$R2P" "$R2U"   ILLUMINACLIP:"$adapter":2:30:10:2:keepBothReads   LEADING:5 TRAILING:15 SLIDINGWINDOW:4:15 MINLEN:40

## 8. Comparar número de reads

In [ ]:
import gzip

def nreads(path):
    with gzip.open(path, "rt") as f:
        return sum(1 for _ in f) // 4

print("R1 bruto:", nreads(R1))
print("R1 paired após trimming:", nreads(R1P))
print("R2 bruto:", nreads(R2))
print("R2 paired após trimming:", nreads(R2P))

## 9. FastQC e MultiQC depois do trimming

In [ ]:
trim_qc = OUT / "fastqc_trimmed"
trim_qc.mkdir(parents=True, exist_ok=True)
!conda run -n bioinfo fastqc -t 2 -o "$trim_qc" "$R1P" "$R2P"

multi_trim = OUT / "multiqc_trimmed"
multi_trim.mkdir(parents=True, exist_ok=True)
!conda run -n bioinfo multiqc "$trim_qc" -o "$multi_trim" -f

## 10. Registrar o ambiente

In [ ]:
ENV_FILE = PASTAS["ambientes"] / "aula04_bioinfo.yml"
!conda env export -n bioinfo --from-history > "$ENV_FILE"
print(ENV_FILE)

## Saída para a próxima aula

A Aula 5 utilizará diretamente:

- `04_qc_trimming/trimmed/hypochilus_petrunkevitchi_SRR15736591_R1_paired.fastq.gz`
- `04_qc_trimming/trimmed/hypochilus_petrunkevitchi_SRR15736591_R2_paired.fastq.gz`